# Validating and auditing a protocol

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do we verify that every gadget circuit and parity equation matches its declared instruction?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Compare a gadget's computed circuit action against its declared objective with `ec.GadgetProfile`.
- Validate a whole protocol with `ec.audit` and interpret error and warning diagnostics.
- Derive missing parity checks, observable readouts, and frame equations automatically with `ec.filled`.

## Setting up

The cell below builds a fresh copy of the baseline `C4` protocol.

In [ ]:
import qodec
from qdk import ec
from _unit import make_baseline_protocol

protocol = make_baseline_protocol()
gadgets = protocol.layers[0].gadgets

## Comparing circuit action against instruction objective

When you write or edit a gadget circuit, how do you know it implements the logical instruction it claims to? `ec.GadgetProfile(gadget)` simulates all measurement branches analytically and exposes two objects:
- `profile.objective`: the logical behavior required by `gadget.implements`.
- `profile.action`: the logical behavior actually realized by `gadget.circuit` and its declared `readouts`.

Calling `profile.action.is_equivalent_to(profile.objective)` checks exact equivalence, while `why_not_equivalent_to` pinpoints any discrepancy. Note that `GadgetProfile` snapshots its input—always create a new `GadgetProfile` after editing a gadget.

In [ ]:
for name in ["prepare_z_all", "cx_all", "measure_z_all"]:
    prof = ec.GadgetProfile(gadgets[name])
    matches = prof.action.is_equivalent_to(prof.objective)
    print(f"{name:<15} -> meets objective: {matches}")

# Demonstrate diagnostic output when control and target blocks are reversed
broken_copy = qodec.Qodec.loads(protocol.dumps())
broken_cx = broken_copy.layers[0].gadgets["cx_all"]
broken_cx.circuit.source = "CX 4 0\nCX 5 1\nCX 6 2\nCX 7 3\n"  # reversed CNOT direction!
broken_prof = ec.GadgetProfile(broken_cx)
print("\nDiagnostic for reversed cx_all circuit:")
print(broken_prof.objective.why_not_equivalent_to(broken_prof.action))

## Auditing a qodec and deriving missing equations

To check an entire protocol at once—code algebra, instruction contracts, circuit calls, gadget actions, checks, readouts, flags, and frames—call `ec.audit(protocol)`.

Writing parity equations by hand is tedious and error-prone. Let's clear the observable readouts on `measure_x_all` and the stabilizer checks on `cx_all` to see the exact `audit` diagnostics, then use `ec.filled` to derive them automatically from noiseless simulation:

In [ ]:
draft = qodec.Qodec.loads(protocol.dumps())
draft.layers[0].gadgets["measure_x_all"].readouts = []
draft.layers[0].gadgets["cx_all"].checks = []

draft_report = ec.audit(draft)
print(
    f"Draft report.ok = {draft_report.ok} "
    f"(errors={len(draft_report.errors)}, warnings={len(draft_report.warnings)})"
)
print(draft_report)

completed = ec.filled(draft)
final_report = ec.audit(completed)
print("\nAfter ec.filled(draft):")
print(final_report)

`ec.filled` returns a new `Gadget` or `Qodec` without mutating its input. It derives parity `checks`, observable `readouts`, and `frames`, while preserving any declared `flags`—because noiseless simulation cannot guess which circuit faults a flag is intended to catch.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why did clearing observable readouts produce audit errors while clearing unitary checks produced warnings?</summary>

<div style="padding:0.1em 1em;">

Without observable readouts, `measure_x_all` fails to define the logical outcomes promised by its `Observe` action (`gadget/missing-observable` error). Without checks on `cx_all`, the unitary circuit still performs the logical CNOT, so `audit` issues `gadget/incomplete-output-frame` warnings (`report.ok` checks errors only unless `promote_warnings=True` is passed).

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Which gadget equations can ec.filled derive automatically, and which must you still declare?</summary>

<div style="padding:0.1em 1em;">

`ec.filled` derives stabilizer `checks`, observable `readouts`, and `frames` from exact noiseless simulation. You must still declare `flags` (on the instruction and in `gadget.readouts`) because flags exist to herald noisy faults rather than noiseless observables.

</div>

</details></div>

## Exercise: Restore missing syndrome checks

Clear `checks` on the baseline `syndrome` gadget, use `ec.filled` to derive its parity equations automatically, and return the number of derived checks.

In [ ]:
from qdk import ec
from _unit import exercise, make_baseline_protocol


@exercise
def repair_syndrome_checks() -> int:
    syn = make_baseline_protocol().layers[0].gadgets['syndrome']
    syn.checks = []
    # Use ec.filled(syn) to derive the missing checks and return len(repaired.checks)
    return ...

**Hint**

Call `repaired = ec.filled(syn)` and return `len(repaired.checks)`.

In [ ]:
from qdk import ec
from _unit import exercise, make_baseline_protocol


@exercise
def repair_syndrome_checks() -> int:
    syn = make_baseline_protocol().layers[0].gadgets['syndrome']
    syn.checks = []
    repaired = ec.filled(syn)
    return len(repaired.checks)

`4`. `ec.filled` derives two ancilla-readout stabilizer checks (`in[0]` to `circuit.readouts`) and two block-conservation checks (`in[0]` to `out[0]`).

## Further reading

- [Audit the protocol and derive missing equations ↗](https://learn.microsoft.com/en-us/azure/quantum/develop-validate-qec-protocol#audit-the-protocol)
- [Validation concepts in qodec ↗](https://github.com/microsoft/qdk-ec/blob/main/qodec/docs/concepts/validation.md)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../03-gadget-anatomy/gadget_anatomy.workbook.ipynb" title="Previous unit: Inspecting Gadgets, Checks, and Readouts" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Inspecting Gadgets, Checks, and Readouts</span></span></a><a href="../05-circuit-distance/circuit_distance.workbook.ipynb" title="Next unit: Analyzing Gadget Distance and Circuit Faults" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Analyzing Gadget Distance and Circuit Faults</span></span><span aria-hidden="true">&#8250;</span></a></div>